In [1]:
import torch

print("1. CUDA 是否可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("2. 抓到的顯示卡名稱:", torch.cuda.get_device_name(0))
else:
    print("2. 系統找不到 NVIDIA 顯示卡，目前只能用 CPU 運算。")

1. CUDA 是否可用: True
2. 抓到的顯示卡名稱: NVIDIA GeForce RTX 4070 SUPER


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import logging
import time

logging.basicConfig(level=logging.INFO, format='%(message)s')

# ==========================================
# 1. 載入 OLMo 模型 (假設你已經在記憶體中載入完成)

print("CUDA 支援開啟狀態:", torch.cuda.is_available())
model_name = "allenai/OLMo-7B-Instruct-hf"  # 建議使用 Instruct 版本以聽從指令
logging.info(f"⏳ 正在載入模型 {model_name} (這可能需要幾分鐘與較大的記憶體)...")

# 1. 設定 4-bit 量化參數
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16, # 運算時保持 bfloat16 精度
    bnb_4bit_use_double_quant=True,        # 雙重量化，進一步節省記憶體
    bnb_4bit_quant_type="nf4"              # 標準的常態分佈 4-bit 格式
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}  # <--- 關鍵修改
)
model.eval()
logging.info("✅ 模型載入完成！")

def generate_advisor_question(target_dimension, conversation_history):
    """將待釐清的維度交給 OLMo，生成對應的自然情境問句"""
    
    prompt = f"""<|system|>
You are a professional quantitative financial advisor. Your goal is to ask a single situational question based on the "Target Dimension" to elicit the user's preference.

[Strict Constraints - Violation will cause system failure]
1. NEVER provide specific investment advice or recommend any ticker symbols.
2. Ask exactly ONE situational question. Keep it strictly under 40 words.
3. NEVER list options (Do NOT use 1. 2. 3. or A. B. C. etc.).
4. Provide ONLY the question itself. No greetings, no explanations, no filler words.
5. The output must end with a question mark (?).

[Example]
Target Dimension: "Historical Return vs. Volatility"
Advisor: Are you willing to endure a potential 20% short-term drop in your portfolio value to pursue higher long-term returns?

Target Dimension: "Volatility vs. Expense Ratio"
Advisor: Would you prefer paying a slightly higher annual management fee to invest in a fund that offers better downside protection during market crashes?

<|user|>
Conversation History:
{conversation_history}

Target Dimension: "{target_dimension}"
Please generate your question:
<|assistant|>
"""
    
    # 將文字轉為 Token 向量
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 執行推論
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=80,       # 因為限制了一句話，80 個 token 絕對夠用
            temperature=0.4,         # 稍微提高一點點，避免它每次都講一模一樣的話
            repetition_penalty=1.15, # 🚨 核心修正：加入重複懲罰，防止它一直鬼打牆講「現金流量」
            do_sample=True,
            top_p=0.9
        )
        
    # 解碼並裁切出回覆
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    response = response.split("待釐清")[0].strip()
    return response

# ==========================================
# 2. 核心測試迴圈 (模擬完整系統狀態機)
# ==========================================
def run_interactive_simulation():
    # 我們只聚焦在三個維度
    dimensions = ["Historical Return vs Volatility", "Historical Return vs Expense Ratio", "Volatility vs Expense Ratio"]
    
    # 初始不確定性設定 (1.0 代表完全不確定)
    uncertainties = {dim: 1.0 for dim in dimensions}
    threshold = 0.3 # 閾值設定
    
    # 1. 輸入基本的設定，替 olmo 擬定基本開場白
    conversation_history = "Agent: Hi! I am your AI financial agent. I will try to understand your preference of investing during our conversation. First of all, could you please introduce yourself?"
    
    print("\n" + "="*60)
    print(" 🤖 智能理財顧問對話測試開始 (輸入 'quit' 結束)")
    print("="*60)
    print(conversation_history)
    
    turn = 1
    while True:
        # 找出目前不確定性最高的維度
        max_dim = max(uncertainties, key=uncertainties.get)
        max_uncert = uncertainties[max_dim]
        
        # 6. 若不確定性低於閾值，讓系統說出結束台詞
        if max_uncert < threshold:
            print("\n🎉 [系統判定] 所有維度的不確定性皆已低於閾值！")
            print("Agent：Thank you very much! I have clearly understand your preference. Next, the system will evaluate the optimal portfolio for you!")
            break
            
        print(f"\n--- [內部狀態] 第 {turn} 輪 | 待釐清維度: {max_dim} (目前不確定性: {max_uncert:.2f}) ---")
        
        # 2. olmo 提出問題
        advisor_question = generate_advisor_question(max_dim, conversation_history)
        print(f"Agent：{advisor_question}")
        
        # 將問題包進歷史對話
        conversation_history += f"\nAgent：{advisor_question}"
        
        # 3. 使用者回覆
        user_reply = input("User：")
        if user_reply.lower() == 'quit':
            break
            
        # 將回覆包進歷史對話
        conversation_history += f"\nUser：{user_reply}"
        
        # 4. 寫好一個隨著次數遞減的隨機變數 (模擬貝氏更新)
        # 假設使用者每次回答，都能讓該問題的不確定性下降 0.2 ~ 0.5
        drop_amount = random.uniform(0.2, 0.5)
        uncertainties[max_dim] = max(0.0, uncertainties[max_dim] - drop_amount)
        
        # (額外擴充) 模擬神經網路捕捉隱藏關聯：回答一題，其他維度也會稍微收斂
        for dim in uncertainties:
            if dim != max_dim:
                uncertainties[dim] = max(0.0, uncertainties[dim] - random.uniform(0.0, 0.15))
        
        turn += 1
    print(conversation_history)
    return conversation_history

# 執行測試
if __name__ == "__main__":
    conversation_history = run_interactive_simulation()

⏳ 正在載入模型 allenai/OLMo-7B-Instruct-hf (這可能需要幾分鐘與較大的記憶體)...


CUDA 支援開啟狀態: True


HTTP Request: HEAD https://huggingface.co/allenai/OLMo-7B-Instruct-hf/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/allenai/OLMo-7B-Instruct-hf/d1c8d0403b2a78a41d52da4d702a9d60976e2ac4/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/allenai/OLMo-7B-Instruct-hf/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/allenai/OLMo-7B-Instruct-hf/d1c8d0403b2a78a41d52da4d702a9d60976e2ac4/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/allenai/OLMo-7B-Instruct-hf/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/allenai/OLMo-7B-Instruct-hf/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/allenai/OLMo-7B-Instruct-hf/resolve/main/c


 🤖 智能理財顧問對話測試開始 (輸入 'quit' 結束)
Hi! I am your AI financial agent. I will try to understand your preference of investing during our conversation. First of all, could you please introduce yourself?

--- [內部狀態] 第 1 輪 | 待釐清維度: Historical Return vs Volatility (目前不確定性: 1.00) ---
Agent：What is more important to you in your investments, maintaining stability in your portfolio's value or experiencing greater gains but potentially facing larger price fluctuations?

--- [內部狀態] 第 2 輪 | 待釐清維度: Volatility vs Expense Ratio (目前不確定性: 0.90) ---
Agent：In exchange for accepting a higher level of volatility, would you be willing to pay a slightly higher annual management fee to invest in a fund with lower expense ratios, aiming for better long-term performance despite possible temporary losses?

--- [內部狀態] 第 3 輪 | 待釐清維度: Historical Return vs Expense Ratio (目前不確定性: 0.81) ---
Agent：Would you be willing to accept slightly higher risks for potentially higher long-term returns by investing in a fund with a high

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import logging
'''
Sanity test for OLMo. Check if it can communitcate fluently without serious hallucination.
'''
# 關閉不必要的 transformers 警告訊息，讓終端機畫面乾淨
logging.getLogger("transformers").setLevel(logging.ERROR)

# 1. 設定 4-bit 量化參數
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16, # 運算時保持 bfloat16 精度
    bnb_4bit_use_double_quant=True,        # 雙重量化，進一步節省記憶體
    bnb_4bit_quant_type="nf4"              # 標準的常態分佈 4-bit 格式
)

# 2. 載入 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. 載入模型 (加入 quantization_config)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}  # <--- 關鍵修改
)
print("✅ 載入完成！我們開始聊天吧 (輸入 'quit' 結束對話)。")
print("-" * 50)

while True:
    user_input = input("\n你：")
    if user_input.lower() in ['quit', 'exit']:
        print("結束對話，系統關閉。")
        break

    # OLMo Instruct 專用的提示詞格式 (Prompt Template)
    # 必須使用 <|user|> 和 <|assistant|> 標籤，模型才會知道現在輪到它講話
    prompt = f"<|user|>\n{user_input}\n<|assistant|>\n"

    # 將文字轉為 Token 並送進 GPU (或自動分配的設備)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 執行推論
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,   # 設定單次回覆的最大長度
            temperature=0.7,      # 0.7 適合一般對話，具備一點創造力但不會亂講話
            do_sample=True,
            top_p=0.9
        )

    # 切割輸出：把前面我們餵進去的 prompt 扣掉，只保留模型新生成的回覆
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    print(f"OLMo：{response}")

Loading weights: 100%|██████████| 226/226 [00:20<00:00, 11.06it/s]


✅ 載入完成！我們開始聊天吧 (輸入 'quit' 結束對話)。
--------------------------------------------------
OLMo：您好！作为一名AI助手，我被設計成能夠快速回答您的問題和提供相關資訊。我的目標是提供準確、准确、有價值的回答。我總是勤奮學習，以提供最新的知識和技能。謝謝您對我的關注！如果您有任何需要，請隨時告訴我。我會盡力為您提供幫助。
OLMo：是的，我對繁體中文有基本的了解。繁體中文是中國大陸和越南的語言，其語言結構和英文不同，需要使用翻譯工具或經驗豐富的翻譯者進行翻譯。繁體中文在文學、故事、電影、網路、銷售、政治等領域都有廣泛的應用，因此有了解繁體中文的基礎知識，對翻譯工作和理解文化有所幫助。
OLMo：我是一個基於訓練的大型語言模型，並且且在這方面有著類似的了解。財務金融是指研究金融系統中金融機構和市場的行為和策略，以及金融系統如何影響社會和經濟發展。這包括了解金融市場、金融工具和金融機構，以及研究金融風險、金融風險管理、金融分析和金融管理。這些主題包括：

*
OLMo：ETFs (Exchange-Traded Funds) 是一種交易工具，它們與基金相似，但它們有一些獨特的特點。下面是一些 ETFs 相關的了解：

1. 市場區分：ETFs 與基金類似，但它們可以在交易所上市，這意味著您可以在任何時刻進行交易。基金必須在每天的市場開盤時上市。
2. 交易時間：您可以在任何時刻進行交易，而基金的交易時間是每天的市場開盤時。
3. 市場份風�
OLMo：是的，我可以對財務金融評估維度提出相關問題，並詢問另一個人對於維度之間的偏好。我會盡力理解和認真回答你的問題，並且針對不同維度進行評估和分析。同時，我也會注意到不同人對於維度的偏好和看法，並且盡力提供相應的建議和解決方案。
結束對話，系統關閉。
